In [1]:
# =====================================================================
# 📦 Cell 1 & 2: 필수 패키지 설치 및 임포트
# =====================================================================
# 로컬 터미널에서 최신 GPU 인식을 위해 아래 명령어를 먼저 실행해 주세요.
%pip install pykrx ta scikit-learn lightgbm yfinance optuna matplotlib seaborn -q
%pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128 --no-cache-dir
%pip install -U FinanceDataReader 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings, pickle
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 운영체제별 폰트 예외 처리 (Windows / Linux)
try:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    font_prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = font_prop.get_name()
except:
    plt.rcParams['font.family'] = 'Malgun Gothic'  # 로컬 윈도우 기본 폰트
plt.rcParams['axes.unicode_minus'] = False

from pykrx import stock
import yfinance as yf
from datetime import datetime, timedelta

from ta.momentum import RSIIndicator, StochasticOscillator
from ta.trend import MACD, EMAIndicator, SMAIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, f1_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb

# 🔥 PyTorch 핵심 컴포넌트 임포트
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# 🔥 로컬 GPU(RTX 5060 Ti) 장치 고정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ PyTorch    : {torch.__version__}')
print(f'✅ LightGBM   : {lgb.__version__}')
print(f'✅ Optuna     : {optuna.__version__}')
print(f'✅ GPU Device : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"})')

# =====================================================================

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement FinanceDataReader (from versions: none)
ERROR: No matching distribution found for FinanceDataReader

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
✅ PyTorch    : 2.12.0.dev20260408+cu128
✅ LightGBM   : 4.6.0
✅ Optuna     : 4.9.0
✅ GPU Device : cuda (NVIDIA GeForce RTX 5060 Ti)


In [ ]:
# =====================================================================
# ⚙️ Cell 3: 설정값 지정 및 필수 라이브러리 로드 (국장 전용)
# =====================================================================
import datetime
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from pykrx import stock
import yfinance as yf
import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, roc_curve
from ta.momentum import RSIIndicator
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.volatility import AverageTrueRange, BollingerBands
from ta.volume import OnBalanceVolumeIndicator

# GPU 사용 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 🎯 타겟 종목 설정 (기아: '000270' | 삼성전자: '005930')
TICKER     = '000270'  
START_DATE = '20150101'
END_DATE   = datetime.datetime.today().strftime('%Y%m%d')

SEQ_LEN        = 60
PRED_DAYS      = 3      # 🎯 3일 뒤 주가 방향성 예측
EPOCHS         = 150    # 최종 모델 학습 에폭 수
TEST_RATIO     = 0.2
LSTM_UNITS = [128, 64] # 첫 번째 LSTM층 128개, 두 번째 64개 뉴런 사용
VAL_RATIO      = 0.1

# 앙상블 가중치 리밸런싱
W_LSTM = 0.05
W_LGBM = 0.65
W_RF   = 0.30

print(f'\n📌 국장 종목 코드 : {TICKER}')
print(f'📅 기간            : {START_DATE} ~ {END_DATE}')
print(f'🔮 예측 타겟       : 미래 {PRED_DAYS}영업일 뒤 방향성')


# =====================================================================
# 📥 Cell 4 & 5: 국장 데이터 + 수급 + 거시경제 융합 및 피처 엔지니어링
# =====================================================================
def fetch_krx_data(ticker, start, end):
    print(f'📡 {ticker} KRX 주가 및 외국인/기관 수급 데이터 수집 중...')
    name = stock.get_market_ticker_name(ticker)
    df = stock.get_market_ohlcv_by_date(start, end, ticker).iloc[:, :5]
    df.columns = ['Open','High','Low','Close','Volume']
    df.index = pd.to_datetime(df.index)
    
    try:
        trade = stock.get_market_trading_volume_by_date(start, end, ticker)
        if '외국인합계' in trade.columns: df['Foreign_Buy'] = trade['외국인합계']
        if '기관합계' in trade.columns: df['Inst_Buy'] = trade['기관합계']
    except:
        df['Foreign_Buy'], df['Inst_Buy'] = 0, 0
    df.dropna(inplace=True)
    return df, name

def fetch_external_data(start, end):
    print('📡 글로벌 거시경제 지수(S&P500, VIX, 환율, 유가) 동기화 중...')
    start_fmt = pd.to_datetime(start, format='%Y%m%d').strftime('%Y-%m-%d')
    end_fmt   = pd.to_datetime(end,   format='%Y%m%d').strftime('%Y-%m-%d')
    tickers = {'SP500':'^GSPC', 'VIX':'^VIX', 'DXY':'DX-Y.NYB', 'OIL':'CL=F'}
    ext = pd.DataFrame()
    for name, sym in tickers.items():
        try:
            ext[name] = yf.download(sym, start=start_fmt, end=end_fmt, progress=False)['Close']
        except:
            print(f'  ⚠️ {name} 다운로드 실패')
    ext.index = pd.to_datetime(ext.index)
    if isinstance(ext.columns, pd.MultiIndex):
        ext.columns = ext.columns.get_level_values(0)
    return ext

def add_features(df):
    d = df.copy()
    d['Return'] = d['Close'].pct_change()
    for i in [2, 5, 10, 20]: d[f'Return_{i}d'] = d['Close'].pct_change(i)
    d['HL_ratio'] = (d['High'] - d['Low']) / d['Close']
    d['OC_ratio'] = (d['Close'] - d['Open']) / (d['Open'] + 1e-9)
    d['Gap'] = (d['Open'] - d['Close'].shift(1)) / (d['Close'].shift(1) + 1e-9)

    for w in [5, 10, 20, 60, 120]:
        sma = SMAIndicator(d['Close'], window=w).sma_indicator()
        d[f'SMA_{w}'] = sma
        d[f'SMA_dist_{w}'] = (d['Close'] - sma) / (sma + 1e-9)
        
    d['EMA_12'] = EMAIndicator(d['Close'], window=12).ema_indicator()
    d['EMA_26'] = EMAIndicator(d['Close'], window=26).ema_indicator()
    macd = MACD(d['Close'])
    d['MACD'], d['MACD_signal'], d['MACD_diff'] = macd.macd(), macd.macd_signal(), macd.macd_diff()
    
    for w in [7, 14, 21]: d[f'RSI_{w}'] = RSIIndicator(d['Close'], window=w).rsi()
    
    d['Volume_ratio'] = d['Volume'] / (d['Volume'].rolling(20).mean() + 1e-9)
    d['Vol_5d'], d['Vol_20d'] = d['Return'].rolling(5).std(), d['Return'].rolling(20).std()
    d['Vol_ratio'] = d['Vol_5d'] / (d['Vol_20d'] + 1e-9)

    if 'Foreign_Buy' in d.columns:
        d['Foreign_MA5'], d['Foreign_MA20'] = d['Foreign_Buy'].rolling(5).mean(), d['Foreign_Buy'].rolling(20).mean()
    if 'Inst_Buy' in d.columns:
        d['Inst_MA5'], d['Inst_MA20'] = d['Inst_Buy'].rolling(5).mean(), d['Inst_Buy'].rolling(20).mean()

    threshold = 0.01  # 1% 기준
    d['Future_Return'] = (d['Close'].shift(-PRED_DAYS) - d['Close']) / d['Close']
    d['Label'] = (d['Future_Return'] > threshold).astype(int)
    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    return d.ffill().bfill().dropna()

# 데이터 로드 파이프라인 가동
df_raw, stock_name = fetch_krx_data(TICKER, START_DATE, END_DATE)
df_ext = fetch_external_data(START_DATE, END_DATE)
df_merged = df_raw.join(df_ext, how='left').ffill().dropna()
df_feat = add_features(df_merged)
print(f"✅ 국장 데이터 및 피처 엔지니어링 융합 완료! 데이터 크기: {df_feat.shape}")


# =====================================================================
# ⚙️ Cell 6: 전처리, 스케일링 및 고정 데이터 분할
# =====================================================================
exclude = ['Label','Open','High','Low','Close','Volume','SP500','VIX','DXY','OIL','Future_Return']
feature_cols = [c for c in df_feat.columns if c not in exclude]

X_raw, y_raw = df_feat[feature_cols].values, df_feat['Label'].values
X_raw = np.where(np.isinf(X_raw), np.nan, X_raw)
X_raw = pd.DataFrame(X_raw, columns=feature_cols).ffill().bfill().values

n = len(X_raw)
test_size, val_size = int(n * TEST_RATIO), int(n * VAL_RATIO)
train_size = n - test_size - val_size

scaler = RobustScaler()
X_tr_sc = scaler.fit_transform(X_raw[:train_size])
X_vl_sc = scaler.transform(X_raw[train_size:train_size+val_size])
X_te_sc = scaler.transform(X_raw[train_size+val_size:])

y_train, y_val, y_test = y_raw[:train_size], y_raw[train_size:train_size+val_size], y_raw[train_size+val_size:]

rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_tr_sc, y_train)
selector = SelectFromModel(rf_selector, threshold='mean', prefit=True)

X_tr_sel = selector.transform(X_tr_sc)
X_vl_sel = selector.transform(X_vl_sc)
X_te_sel = selector.transform(X_te_sc)

def make_sequences(X, y, seq_len):
    xs, ys = [], []
    for i in range(seq_len, len(X)):
        xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(xs), np.array(ys)

X_tr, y_tr = make_sequences(X_tr_sel, y_train, SEQ_LEN)
X_vl, y_vl = make_sequences(X_vl_sel, y_val,   SEQ_LEN)
X_te, y_te = make_sequences(X_te_sel, y_test,  SEQ_LEN)
X_tr_flat, X_vl_flat, X_te_flat = X_tr[:, -1, :], X_vl[:, -1, :], X_te[:, -1, :]


# =====================================================================
# 🤖 Cell 7: 🔥 PyTorch BiLSTM + Attention 모델 정의
# =====================================================================
class PyTorchBiLSTMAttention(nn.Module):
    def __init__(self, n_features, seq_len, lstm_units=[128, 64], dropout=0.5):
        super(PyTorchBiLSTMAttention, self).__init__()
        self.lstm1 = nn.LSTM(input_size=n_features, hidden_size=lstm_units[0], num_layers=1, batch_first=True, bidirectional=True)
        self.bn1 = nn.LayerNorm(lstm_units[0] * 2)
        self.lstm2 = nn.LSTM(input_size=lstm_units[0] * 2, hidden_size=lstm_units[1], num_layers=1, batch_first=True, bidirectional=True)
        self.bn2 = nn.LayerNorm(lstm_units[1] * 2)
        self.dropout_lstm = nn.Dropout(dropout)
        self.att_dense = nn.Linear(lstm_units[1] * 2, 1)
        self.fc1 = nn.Linear(lstm_units[1] * 2, 64)
        self.dropout1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 32)
        self.dropout2 = nn.Dropout(dropout)
        self.out_dense = nn.Linear(32, 1)
        
    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.dropout_lstm(x)
        x = self.bn1(x)
        x, _ = self.lstm2(x)
        x = self.dropout_lstm(x)
        x = self.bn2(x)
        score = torch.tanh(self.att_dense(x))     
        weights = F.softmax(score, dim=1)        
        context = x * weights                     
        context = torch.mean(context, dim=1)      
        x = F.relu(self.fc1(context))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        out = torch.sigmoid(self.out_dense(x))
        return out


# =====================================================================
# 🏋️ Cell 8: PyTorch 모델 훈련 함수 구체화
# =====================================================================
def train_pytorch_engine(model, X_train, y_train, X_val, y_val, epochs, batch_size, lr, class_weight, patience=15):
    X_tr_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
    X_vl_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    
    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    criterion = nn.BCELoss(reduction='none')
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    
    best_auc, patience_counter = 0.0, 0
    best_model_state = None
    history = {'auc': [], 'val_auc': []}
    w0, w1 = class_weight[0], class_weight[1]
    
    for epoch in range(epochs):
        model.train()
        train_probs, train_targets = [], []
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss_elements = criterion(outputs, batch_y)
            weights = batch_y * w1 + (1 - batch_y) * w0
            loss = (loss_elements * weights).mean()
            loss.backward()
            optimizer.step()
            train_probs.extend(outputs.detach().cpu().numpy())
            train_targets.extend(batch_y.cpu().numpy())
            
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_vl_t).cpu().numpy()
            
        train_auc = roc_auc_score(train_targets, train_probs) if len(np.unique(train_targets)) > 1 else 0.5
        val_auc = roc_auc_score(y_val, val_outputs) if len(np.unique(y_val)) > 1 else 0.5
        history['auc'].append(train_auc)
        history['val_auc'].append(val_auc)
        scheduler.step(val_auc)
        
        if val_auc > best_auc:
            best_auc = val_auc
            patience_counter = 0
            best_model_state = pickle.loads(pickle.dumps(model.state_dict()))
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
                
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return history


# =====================================================================
# 🔥 [친구분 조언 반영 핵심] Sliding Window + 파라미터 Grid Search 엔진
# =====================================================================
# =====================================================================
# 🚀 [수정된 Cell 8] 최적화 파라미터 탐색 (로그 최소화 버전)
# =====================================================================
print("\n🚀 [Grid Search] 파라미터 최적화 및 슬라이딩 윈도우 검증 시작...")
print("    (잠시만 기다려 주세요. 최적값을 찾은 후 결과를 출력합니다.)")

batch_sizes = [32, 64, 128]
learning_rates = [0.01, 0.001, 0.0001]

best_overall_score = -1.0
best_hyperparams = {'batch_size': 64, 'lr': 0.001}

# 루프 내부의 print문들을 전부 제거하여 깔끔하게 수행
for b_size in batch_sizes:
    for l_rate in learning_rates:
        fold_scores = []
        total_len = len(df_feat)
        n_splits = 5
        fold_size = total_len // (n_splits + 1)
        
        for fold in range(n_splits):
            train_end = fold_size * (fold + 1)
            test_end  = min(train_end + fold_size, total_len)
            if test_end - train_end < 10: continue
            
            df_train_fold = df_feat.iloc[:train_end]
            df_test_fold = df_feat.iloc[train_end:test_end]
            
            v_idx = int(len(df_train_fold) * 0.9)
            Xtr_raw = df_train_fold.iloc[:v_idx][feature_cols].values
            ytr_fold = df_train_fold.iloc[:v_idx]['Label'].values
            Xvl_raw = df_train_fold.iloc[v_idx:][feature_cols].values
            yvl_fold = df_train_fold.iloc[v_idx:]['Label'].values
            Xte_raw = df_test_fold[feature_cols].values
            yte_fold = df_test_fold['Label'].values
            
            if len(Xtr_raw) <= SEQ_LEN or len(Xvl_raw) <= SEQ_LEN or len(Xte_raw) <= SEQ_LEN: continue
                
            f_scaler = RobustScaler()
            Xtr_sc = f_scaler.fit_transform(Xtr_raw)
            Xvl_sc = f_scaler.transform(Xvl_raw)
            Xte_sc = f_scaler.transform(Xte_raw)
            
            Xtr_s, ytr_s = make_sequences(Xtr_sc[:, :X_tr.shape[2]], ytr_fold, SEQ_LEN)
            Xvl_s, yvl_s = make_sequences(Xvl_sc[:, :X_tr.shape[2]], yvl_fold, SEQ_LEN)
            Xte_s, yte_s = make_sequences(Xte_sc[:, :X_tr.shape[2]], yte_fold, SEQ_LEN)
            
            if len(Xtr_s) == 0 or len(Xvl_s) == 0 or len(Xte_s) == 0: continue
            if len(np.unique(ytr_s)) < 2 or len(np.unique(yvl_s)) < 2 or len(np.unique(yte_s)) < 2: continue
            
            cw_f = compute_class_weight('balanced', classes=np.array([0,1]), y=ytr_s)
            class_weight_fold = {0: cw_f[0], 1: cw_f[1]}
            
            test_model = PyTorchBiLSTMAttention(Xtr_s.shape[2], SEQ_LEN, LSTM_UNITS, dropout=0.5).to(device)
            
            _ = train_pytorch_engine(
                test_model, Xtr_s, ytr_s, Xvl_s, yvl_s, 
                epochs=15, batch_size=b_size, lr=l_rate, 
                class_weight=class_weight_fold, patience=5
            )
            
            test_model.eval()
            with torch.no_grad():
                preds = test_model(torch.tensor(Xte_s, dtype=torch.float32).to(device)).cpu().numpy().flatten()
            
            fold_scores.append(roc_auc_score(yte_s, preds))
            
        if len(fold_scores) > 0:
            avg_auc = np.mean(fold_scores)
            if avg_auc > best_overall_score:
                best_overall_score = avg_auc
                best_hyperparams['batch_size'] = b_size
                best_hyperparams['lr'] = l_rate

print("\n✅ 최적화 탐색 완료!")
print(f"🎯 최종 확정 - BATCH_SIZE: {best_hyperparams['batch_size']} | LEARNING_RATE: {best_hyperparams['lr']}")
print(f"📊 최고 평균 검증 AUC: {best_overall_score:.4f}\n")

# 찾은 최적 가중치를 기반으로 메인 모델 학습 세팅 고정
FINAL_BATCH_SIZE = best_hyperparams['batch_size']
FINAL_LEARNING_RATE = best_hyperparams['lr']


# =====================================================================
# 🏋️ Cell 8-2: 최적화된 파라미터 기반 메인 정밀 훈련 시작
# =====================================================================
print('🏋️ 최적화된 스펙으로 최종 정밀 학습 엔진 가동...')
lstm_model = PyTorchBiLSTMAttention(X_tr.shape[2], SEQ_LEN, LSTM_UNITS, dropout=0.5).to(device)
cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
class_weight = {0: cw[0], 1: cw[1]}
history_dict = train_pytorch_engine(lstm_model, X_tr, y_tr, X_vl, y_vl, EPOCHS, FINAL_BATCH_SIZE, FINAL_LEARNING_RATE, class_weight, patience=25)
print('✅ PyTorch LSTM 모델 트레이닝 완료!')


# =====================================================================
# 🌲 Cell 9 & 10: LightGBM + Optuna 자동 튜닝 및 RandomForest 학습
# =====================================================================
print('\n🔍 Optuna 기반 LightGBM 하이퍼파라미터 베이지안 탐색 중...')
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'num_leaves': trial.suggest_int('num_leaves', 20, 60),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'class_weight': 'balanced', 'random_state': 42, 'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr_flat, y_tr, eval_set=[(X_vl_flat, y_vl)], callbacks=[lgb.early_stopping(20, verbose=False)])
    return roc_auc_score(y_vl, model.predict_proba(X_vl_flat)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10, show_progress_bar=False)

lgbm_model = lgb.LGBMClassifier(**study.best_params, class_weight='balanced', random_state=42, verbose=-1)
lgbm_model.fit(X_tr_flat, y_tr, eval_set=[(X_vl_flat, y_vl)], callbacks=[lgb.early_stopping(30, verbose=False)])
print(f'✅ LightGBM 최적화 완료! (Best AUC: {study.best_value:.4f})')

print('🌳 RandomForest 학습 프로세스 시작...')
rf_model = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_split=10, min_samples_leaf=5, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_tr_flat, y_tr)
print('✅ RandomForest 최종 최적화 완료!')


# =====================================================================
# 🎯 Cell 11: 앙상블 시스템 (Soft Voting 연산)
# =====================================================================
print('\n🎯 앙상블 시스템 결합 및 최종 예측 테스트 중...')
lstm_model.eval()
with torch.no_grad():
    X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)
    lstm_prob = lstm_model(X_te_t).cpu().numpy().flatten()

lgbm_prob = lgbm_model.predict_proba(X_te_flat)[:, 1]
rf_prob   = rf_model.predict_proba(X_te_flat)[:, 1]

ensemble_prob = W_LSTM * lstm_prob + W_LGBM * lgbm_prob + W_RF * rf_prob
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

print('='*55)
print(f'   📋 [{stock_name}] 모델별 테스트 스코어 최종 스펙')
print('='*55)
for name, pred, prob in [
    ('LSTM (PyTorch)', (lstm_prob >= 0.5).astype(int), lstm_prob),
    ('LightGBM      ', (lgbm_prob >= 0.5).astype(int), lgbm_prob),
    ('RandomForest ', (rf_prob >= 0.5).astype(int), rf_prob),
    ('🏆 앙상블 모델 ', ensemble_pred, ensemble_prob),
]:
    print(f'   {name} | Acc={accuracy_score(y_te, pred):.4f} | AUC={roc_auc_score(y_te, prob):.4f} | F1={f1_score(y_te, pred, zero_division=0):.4f}')
print('='*55)




📌 국장 종목 코드 : 000270
📅 기간            : 20150101 ~ 20260623
🔮 예측 타겟       : 미래 3영업일 뒤 방향성
📡 000270 KRX 주가 및 외국인/기관 수급 데이터 수집 중...
Error occurred in get_market_trading_value_and_volume_on_ticker_by_date: Expecting value: line 1 column 1 (char 0)
📡 글로벌 거시경제 지수(S&P500, VIX, 환율, 유가) 동기화 중...
✅ 국장 데이터 및 피처 엔지니어링 융합 완료! 데이터 크기: (2815, 41)

🚀 [Grid Search] 파라미터 최적화 및 슬라이딩 윈도우 검증 시작...
    (잠시만 기다려 주세요. 최적값을 찾은 후 결과를 출력합니다.)

✅ 최적화 탐색 완료!
🎯 최종 확정 - BATCH_SIZE: 64 | LEARNING_RATE: 0.001
📊 최고 평균 검증 AUC: 0.5702

🏋️ 최적화된 스펙으로 최종 정밀 학습 엔진 가동...
✅ PyTorch LSTM 모델 트레이닝 완료!

🔍 Optuna 기반 LightGBM 하이퍼파라미터 베이지안 탐색 중...
✅ LightGBM 최적화 완료! (Best AUC: 0.5865)
🌳 RandomForest 학습 프로세스 시작...
✅ RandomForest 최종 최적화 완료!

🎯 앙상블 시스템 결합 및 최종 예측 테스트 중...
   📋 [기아] 모델별 테스트 스코어 최종 스펙
   LSTM (PyTorch) | Acc=0.6083 | AUC=0.5076 | F1=0.2836
   LightGBM       | Acc=0.5268 | AUC=0.5363 | F1=0.4516
   RandomForest  | Acc=0.5189 | AUC=0.5079 | F1=0.4292
   🏆 앙상블 모델  | Acc=0.5229 | AUC=0.5221 | F1=0.4175
📡 000270 KRX 주가 및 외국인/기

In [ ]:
# =====================================================================
# 🔮 Cell 14 & 15: 실전 최신 국장 주가 방향 예측 대시보드
# =====================================================================
def predict_latest_krx(ticker=TICKER):
    df_new, nm = fetch_krx_data(ticker, START_DATE, END_DATE)
    df_ext_new = fetch_external_data(START_DATE, END_DATE)
    df_f = add_features(df_new.join(df_ext_new, how='left').ffill().dropna())

    X_new = np.where(np.isinf(df_f[feature_cols].values), np.nan, df_f[feature_cols].values)
    X_new = pd.DataFrame(X_new, columns=feature_cols).ffill().bfill().values

    X_new_sc = scaler.transform(X_new)
    X_new_sel = selector.transform(X_new_sc)

    seq  = X_new_sel[-SEQ_LEN:][np.newaxis, ...]
    flat = X_new_sel[-1:, :]

    lstm_model.eval()
    with torch.no_grad():
        seq_t = torch.tensor(seq, dtype=torch.float32).to(device)
        lp = lstm_model(seq_t).cpu().numpy()[0][0]
        
    gp = lgbm_model.predict_proba(flat)[0][1]
    rp = rf_model.predict_proba(flat)[0][1]
    prob = W_LSTM*lp + W_LGBM*gp + W_RF*rp

    direction = '📈 상승 시그널 포착' if prob >= 0.5 else '📉 하락 시그널 포착'
    conf = max(prob, 1-prob) * 100

    print('='*55)
    print(f'   🔮 PyTorch 앙상블 실전 예측 대시보드: {nm}({ticker})')
    print('='*55)
    print(f'   현재 기준가 : {int(df_f["Close"].iloc[-1]):,}원 ({df_f.index[-1].strftime("%Y-%m-%d")})')
    print(f'   예측 타겟   : 미래 {PRED_DAYS}영업일 뒤 방향성 예측 시뮬레이션 결과')
    print(f'   ---------------------------------------------------')
    print(f'   [모델 1] PyTorch LSTM     : {lp*100:.1f}% 상승 확증')
    print(f'   [모델 2] LightGBM Boost   : {gp*100:.1f}% 상승 확증')
    print(f'   [모델 3] RandomForest     : {rp*100:.1f}% 상승 확증')
    print(f'   ───────────────────────────────────────────────────')
    print(f'   🏆 최적화 앙상블 합산 결론 : {direction} ({prob*100:.1f}%, 신뢰도 {conf:.1f}%)')
    print('='*55)
    return prob

# 실전 국장 예측 연산 실행
prob_final = predict_latest_krx(TICKER)


# =====================================================================
# 💾 Cell 16: 최적화 가중치 로컬 물리 파일 영구 저장
# =====================================================================
torch.save(lstm_model.state_dict(), 'lstm_model_krx_grid.pt')
lgbm_model.booster_.save_model('lgbm_model_krx_grid.txt')
print('\n💾 [물리 백업] 그리드 서치가 반영된 국장 전용 가중치가 완벽하게 백업되었습니다!')